# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² CRC Survivors Dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset schema is published as a [Croissant](https://mlcommons.org/croissant) JSON-LD file and can be accessed via this URL:
- https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print("Fields available in metadata:")
for attr in dir(metadata):
    if not attr.startswith('_') and not callable(getattr(metadata, attr)):
        print(f"- {attr}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`.

We'll list the record sets (`cr:RecordSet`) in the dataset, then for each one list its fields and columns by `@id`.

In [ ]:
def print_record_sets(ds):
    print('Listing all record sets and (if present) their fields/columns:')
    record_sets = ds.list_record_sets()
    for rs in record_sets:
        print(f"\nRecordSet '@id': {rs['@id']} (name: {rs.get('name', rs['@id'])})")
        fields = ds.list_fields(rs['@id'])
        if fields:
            print("  Fields:")
            for field in fields:
                print(f"    - @id: {field['@id']}  (name: {field.get('name', field['@id'])})  type: {field.get('dataType')}")
        else:
            print("  No fields found.")
        columns = ds.list_columns(rs['@id'])
        if columns:
            print("  Columns:")
            for col in columns:
                print(f"    - @id: {col['@id']}  (name: {col.get('name', col['@id'])})")
        else:
            print("  No columns found.")
    return [rs['@id'] for rs in record_sets]

# List the record sets & examine their fields
record_set_ids = print_record_sets(dataset)

## 3. Data Extraction

Load one or multiple record sets into pandas DataFrames for analysis, using their `@id`.

In [ ]:
# List the record_set_ids discovered in the previous cell:
print("Record sets available:")
for idx, rsid in enumerate(record_set_ids):
    print(f"  [{idx}] {rsid}")

# You can select whichever record set (by '@id'), here we load all of them as DataFrames:
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from RecordSet '@id': {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows. Columns: {df.columns.tolist()}")

# For demonstration, display the first few rows for the first record set
first_record_set = record_set_ids[0]
print(f"\nPreview of first record set ({first_record_set}):")
display(dataframes[first_record_set].head())

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate basic exploratory analysis using one record set. 
First, identify numeric fields to use for filtering/normalization, then show grouping and some summary statistics.

> **Note:** All columns/fields referenced are by their `@id` string, as listed previously.

In [ ]:
# Choose a record set and inspect columns to select numeric fields
record_set_id = first_record_set  # reuse the first one loaded
df = dataframes[record_set_id]
print(f"Columns in {record_set_id}: {df.columns.tolist()}")

# Identify numeric columns (you may need to inspect output and adjust selection)
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields (by @id): {numeric_candidates}")

# If no numeric types detected (e.g. all loaded as object), try manual conversion
if not numeric_candidates:
    print("Attempting to infer numeric columns...")
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            pass
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    print(f"After conversion, numeric fields: {numeric_candidates}")

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # select the first numeric field
else:
    print("No numeric field found for EDA.")
    numeric_field_id = None

if numeric_field_id:
    # Filtering based on a numeric threshold (example: 10 or median+IQR)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
    print(filtered_df[[numeric_field_id]].head())

    # Normalization (z-score)
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a categorical field (pick one that isn't the index/numeric field)
    possible_group_fields = [c for c in df.columns if c != numeric_field_id and not c.endswith('_normalized')]
    group_field_id = None
    for _f in possible_group_fields:
        # Heuristic: choose a string/categorical column
        if df[_f].dtype == object and df[_f].nunique() > 1:
            group_field_id = _f
            break

    if group_field_id:
        print(f"\nGrouping filtered records by '{group_field_id}' and aggregating mean:")
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped.head())
    else:
        print("No suitable group field detected for grouping.")
else:
    print("No numeric field identified in this record set.")

## 5. Visualization

Plot data distributions (such as a histogram of the main numeric field), or counts of key categorical variables, using the `@id` for all references.

<sup>If you wish to explore further, select different record sets and adapt visualizations accordingly.</sup>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True, color='purple')
    plt.title(f"Distribution of field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(9,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("Visualization demo skipped: no numeric field in selected record set.")

## 6. Conclusion

- We loaded and explored the [FAIR² CRC Survivors Dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the Croissant schema and `mlcroissant` tools.
- The notebook demonstrated how to:
    - Load dataset metadata and programmatically list record sets, fields, and columns by their `@id`
    - Import tabular records into pandas DataFrames, referencing data programmatically by `@id`
    - Perform basic EDA, filtering, normalization, grouping, and simple visualizations based on fields' `@id`
- This approach allows robust, reproducible access to complex datasets with complete data provenance and structural clarity, in line with the FAIR² paradigm.

**Next steps:** Data scientists can use this workflow to perform statistical analysis, train ML models, or curate custom analytic views while accurately referencing all data entities by `@id` as required for interoperable, reusable research.